# Notebook 02: ETL & Silver Layer Validation
This notebook validates the Silver Layer data quality after running transformation and cleansing steps.

In [ ]:
import sys
from pathlib import Path
from pyspark.sql import functions as F

sys.path.append('../src')
from utils import create_spark_session, get_project_root

# Cell 1: Load Silver Parquet
spark = create_spark_session(app_name="02-etl-validation")
root_dir = get_project_root()
silver_path = root_dir / "data" / "silver"
df_silver = spark.read.parquet(str(silver_path))
print(f"Loaded Silver dataset from: {silver_path}")

In [ ]:
# Cell 2: Display Schema and Row Count
silver_count = df_silver.count()
print(f"Total Silver Records: {silver_count:,}")
df_silver.printSchema()

In [ ]:
# Cell 3: Check for Nulls in Critical Columns
critical_cols = ["price", "date", "property_type", "new_build", "duration"]
null_exprs = [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in critical_cols]
nulls_df = df_silver.select(null_exprs)
print("Null Counts in Critical Columns:")
nulls_df.show(truncate=False)

In [ ]:
# Cell 4: Display Sample Rows (First 10)
df_silver.show(10, truncate=False)

In [ ]:
# Cell 5: Compare Silver Row Count vs Bronze Row Count
bronze_path = root_dir / "data" / "bronze"
df_bronze = spark.read.parquet(str(bronze_path))
bronze_count = df_bronze.count()
dropped_count = bronze_count - silver_count
retention_rate = (silver_count / bronze_count) * 100

print(f"Bronze Count:    {bronze_count:,}")
print(f"Silver Count:    {silver_count:,}")
print(f"Dropped Records: {dropped_count:,}")
print(f"Retention Rate:  {retention_rate:.2f}%")

In [ ]:
# Cell 6: Statistics for Price Column after Cleaning
price_stats = df_silver.select(
    F.count("price").alias("count"),
    F.mean("price").alias("mean"),
    F.stddev("price").alias("stddev"),
    F.min("price").alias("min"),
    F.max("price").alias("max")
)
price_stats.show(truncate=False)

In [ ]:
# Cell 7: Verify No Negative Prices, No Zero Prices
invalid_prices = df_silver.filter(F.col("price") <= 0).count()
print(f"Records with Price <= 0: {invalid_prices}")
assert invalid_prices == 0, "Validation Error: Non-positive prices found in Silver Layer!"
print("Silver Layer Validation Successful!")